## Image Scraper Demo

Quick notebook to demo a image scraper for the web. Enventually turn into the class to be reused later. This notebook will be to validate that it works.

Initial code: https://brightdata.com/blog/how-tos/scrape-images-from-websites

### Selenium Setup

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import StaleElementReferenceException
import urllib.request

Iniitalize a headless Webdriver instance

In [2]:
options = Options()
# options.add_argument("--headless") # commentted out for dev work

# initialize a chrome webdriver instance with specified optoins
driver = webdriver.Chrome(
    service=ChromeService(),
    options=options
)

Due to pages displaying images differently depending on the screen size of the user device. We will maximize the chrome window.

In [3]:
driver.maximize_window()

### Page Selection and Setup

Instructed chrome to connect to a target page via selenium

In [4]:
url = "https://unsplash.com/s/photos/wallpaper?license=free"

In [5]:
driver.get(url)

#### Retrieve All Images Urls

In [6]:
# image_html_nodes = driver.find_elements(By.CSS_SELECTOR, '[data-testid="asset-grid-masonry-img"]')

# TODO: Add crawling logic to figure the next page

Initializa a list that contain the urls extracted from the image. Then iterate over hte nodes in image_html_nodes, collect the url in the src or the url

In [7]:
image_urls = []

try:
    # moves the load next element
    more_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.CLASS_NAME, 'loadMoreButton-FHXj5M'))
    )
    more_button.click()

    # checks for all images elements
    image_html_nodes = driver.find_elements(By.CSS_SELECTOR, '[data-testid="asset-grid-masonry-img"]')
    total_image_counter = 0
    number_of_images = len(image_html_nodes)

    # image elements load in after users scrolls to them 
    while number_of_images < 1000:
        action = ActionChains(driver)
        action.move_to_element(image_html_nodes[number_of_images-1]).perform()
        image_html_nodes = driver.find_elements(By.CSS_SELECTOR, '[data-testid="asset-grid-masonry-img"]')
        number_of_images = len(image_html_nodes)
        total_image_counter = number_of_images
        print(len(image_html_nodes))
        print(f"total images: {total_image_counter}")
    

    for image_node in image_html_nodes:
        try:
            # use the url in the src as the default behavior
            image_url = image_node.get_attribute("src")

            # extract the url of the larget image from scrset
            srcset = image_node.get_attribute("srcset")

            if srcset is not None:
                # get the last element from the srcset value
                srcset_last_element = srcset.split(",")[-1]

                # get the first element of the value
                image_url = srcset_last_element.split(" ")[1]

            # add the image url to the list
            image_urls.append(image_url)
        except StaleElementReferenceException as e:
            continue

except StaleElementReferenceException:
    pass

60
total images: 60
80
total images: 80
100
total images: 100
120
total images: 120
139
total images: 139
159
total images: 159
179
total images: 179
198
total images: 198
217
total images: 217
237
total images: 237
257
total images: 257
277
total images: 277
296
total images: 296
316
total images: 316
336
total images: 336
356
total images: 356
376
total images: 376
394
total images: 394
414
total images: 414
430
total images: 430
450
total images: 450
470
total images: 470
490
total images: 490
510
total images: 510
529
total images: 529
549
total images: 549
569
total images: 569
589
total images: 589
609
total images: 609
629
total images: 629
649
total images: 649
669
total images: 669
689
total images: 689
709
total images: 709
729
total images: 729
749
total images: 749
769
total images: 769
789
total images: 789
809
total images: 809
829
total images: 829
849
total images: 849
869
total images: 869
889
total images: 889
909
total images: 909
929
total images: 929
949
total imag

In [8]:
print(image_urls[0])

https://images.unsplash.com/photo-1563387852576-964bc31b73af?w=2000&auto=format&fit=crop&q=60&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxzZWFyY2h8MXx8d2FsbHBhcGVyfGVufDB8fDB8fHwy


### Download the Images

Easiest way to download the image in python is use the urlretrieve method from url.request package.

In [9]:
image_name_counter = 1

# download each image and add it to the /images local folder
for url in image_urls:
    print(f"Downloading image no. {image_name_counter}....")

    file_name = f"./images/{image_name_counter}.jpg"

    # address the http error 403 forbidden
    opener = urllib.request.build_opener()
    user_agent_string = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
    opener.addheaders = [("User-Agent", user_agent_string)]
    urllib.request.install_opener(opener)

    # download the image
    urllib.request.urlretrieve(url, file_name)

    print(f"images downlaoded successfully to {file_name}\n")

    # increment the image counter
    image_name_counter += 1

images downlaoded successfully to ./images/1.jpg

images downlaoded successfully to ./images/2.jpg

images downlaoded successfully to ./images/3.jpg

images downlaoded successfully to ./images/4.jpg

images downlaoded successfully to ./images/5.jpg

images downlaoded successfully to ./images/6.jpg

images downlaoded successfully to ./images/7.jpg

images downlaoded successfully to ./images/8.jpg

images downlaoded successfully to ./images/9.jpg

images downlaoded successfully to ./images/10.jpg

images downlaoded successfully to ./images/11.jpg

images downlaoded successfully to ./images/12.jpg

images downlaoded successfully to ./images/13.jpg

images downlaoded successfully to ./images/14.jpg

images downlaoded successfully to ./images/15.jpg

images downlaoded successfully to ./images/16.jpg

images downlaoded successfully to ./images/17.jpg

images downlaoded successfully to ./images/18.jpg

images downlaoded successfully to ./images/19.jpg

images downlaoded successfully to ./imag

Close the browser window 

In [10]:
driver.quit()